In [18]:
import sys
import os

# Add the directory containing the file to the system path
sys.path.append(os.path.abspath("../module1"))
from ingest import load_faq_data
documents = load_faq_data()

In [19]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [20]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [21]:
documents = documents_llm

In [22]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])
type(doc)

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


dict

In [23]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

We want the output as a list of strings, so we define that structure with a Pydantic model:

In [24]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [25]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [26]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [27]:
import json

user_prompt = json.dumps(doc)

In [28]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [29]:
type(user_prompt)

str

In [30]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [31]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [32]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I just found out about it?', 'Is it too late to start this course now?', 'If I join late, can I still get a certificate?', 'What do I need to do to be eligible for the certificate after joining late?', 'Can I submit the project later and still receive the certificate?']


In [33]:
from evaluation_utils import llm_structured

In [34]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I found it late?', 'Is it too late to start the course now?', 'If I join after the course has started, can I still get a certificate?', 'Do I have to submit my project before submissions close to get the certificate?', 'What’s the deadline for project submission if I want the course certificate?']


In [35]:
result

Questions(questions=['Can I still join the course if I found it late?', 'Is it too late to start the course now?', 'If I join after the course has started, can I still get a certificate?', 'Do I have to submit my project before submissions close to get the certificate?', 'What’s the deadline for project submission if I want the course certificate?'])

In [36]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=83, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=290)

**Tracking cost**

In [37]:
usage.input_tokens, usage.output_tokens

(207, 83)

In [38]:
from evaluation_utils import calc_price

In [39]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.00037349999999999997,
 'total_cost': 0.0005287499999999999}

In [40]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start the course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit my project before submissions close to get the certificate?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for project submission if I want the course certificate?',
  'document': '74eb249bbf'}]

In [41]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [42]:
from evaluation_utils import llm_structured_retry

In [43]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [44]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [45]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [49]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [50]:
results

[([{'question': 'I just found this course — can I still join, and is it too late to start?',
    'document': '74eb249bbf'},
   {'question': 'If I join now, do I still have a chance to get the certificate?',
    'document': '74eb249bbf'},
   {'question': 'What do I need to do to be eligible for the certificate if I start late?',
    'document': '74eb249bbf'},
   {'question': 'Can I submit the project after the submission window closes, or does it have to be on time for the certificate?',
    'document': '74eb249bbf'},
   {'question': 'I missed the beginning of the course — can I still participate and earn a certificate?',
    'document': '74eb249bbf'}],
  ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=108, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=315)),
 ([{'question': 'I signed up for the LLM Zoomcamp, but I haven’t got any confirmation email yet — is that a problem?',
    

In [51]:
len(results)

113

In [44]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08688299999999999

In [45]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08688299999999999

In [46]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [48]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)